# 多模態：影像 + 文字 (SigLIP 2 與 Qwen3-VL)

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/06-Vision-Language.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

建議環境：Google Colab，`執行階段 → 變更執行階段類型 → T4 GPU`

In [ ]:
!pip install -q -U transformers
# 下載本章練習的參考解答 (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['q_siglip_zeroshot.py', 'q_vlm_json.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
from urllib.request import urlopen
device = "cuda" if torch.cuda.is_available() else "cpu"
def show(img, width=300):
    display(img.resize((width, int(width * img.height / img.width))))

## SigLIP 2：把影像和文字放進同一個向量空間

用大量「圖片 + 說明文字」訓練，讓相配的圖片和文字的向量接近 (contrastive learning)。
應用：以文搜圖、zero-shot 分類、作為多模態 LLM 的視覺編碼器。

In [ ]:
from transformers import AutoModel, AutoProcessor
siglip_id = "google/siglip2-base-patch16-224"
siglip = AutoModel.from_pretrained(siglip_id, dtype=torch.float16).to(device).eval()
siglip_proc = AutoProcessor.from_pretrained(siglip_id)

### 資料：Oxford-IIIT Pet 的前 300 張測試圖片

用 streaming 模式，不需要下載整個資料集。

In [ ]:
from datasets import load_dataset
pets_stream = load_dataset("timm/oxford-iiit-pet", split="test", streaming=True)
class_names = pets_stream.features["label"].names
samples = list(pets_stream.shuffle(seed=0, buffer_size=1000).take(300))
images = [s["image"].convert("RGB") for s in samples]
labels = np.array([s["label"] for s in samples])
len(images)

In [ ]:
@torch.no_grad()
def image_embeddings(images, batch_size=64):
    embs = []
    for i in range(0, len(images), batch_size):
        inputs = siglip_proc(images=images[i:i + batch_size], return_tensors="pt").to(device)
        out = siglip.get_image_features(**inputs)
        out = out if isinstance(out, torch.Tensor) else out.pooler_output
        embs.append(out.float())
    return torch.nn.functional.normalize(torch.cat(embs), dim=-1)

@torch.no_grad()
def text_embeddings(texts):
    # SigLIP 2 訓練時文字都 padding 到 64 個 token，使用時也要一樣
    inputs = siglip_proc(text=texts, padding="max_length", max_length=64, return_tensors="pt").to(device)
    out = siglip.get_text_features(**inputs)
    out = out if isinstance(out, torch.Tensor) else out.pooler_output
    return torch.nn.functional.normalize(out.float(), dim=-1)

img_emb = image_embeddings(images)
img_emb.shape

### 以文搜圖

In [ ]:
def search(query, k=5):
    scores = (text_embeddings([query]) @ img_emb.T)[0]
    top = scores.topk(k).indices.tolist()
    display(Image.fromarray(np.concatenate([np.array(images[i].resize((160, 160))) for i in top], axis=1)))
search("a black cat")

In [ ]:
search("a dog lying on the grass")

In [ ]:
search("a dog wearing clothes")

### 以圖搜圖

In [ ]:
q = 7
show(images[q], 160)
scores = img_emb @ img_emb[q]
top = scores.topk(6).indices.tolist()[1:]
display(Image.fromarray(np.concatenate([np.array(images[i].resize((160, 160))) for i in top], axis=1)))

## Q：Zero-shot 分類
用 37 個品種名稱組成文字（例如 `"a photo of a {name}, a type of pet."`），計算這 300 張圖的 zero-shot 準確率。

和上一個單元用 3680 張圖片監督式訓練的結果比較看看。

In [ ]:
# %load q_siglip_zeroshot.py
%run -i q_siglip_zeroshot.py

## Vision Language Model (VLM)：Qwen3-VL

VLM = 視覺編碼器 + LLM。圖片被切成小塊 (patch)，轉成「視覺 token」，和文字 token 一起送進 LLM。

[Qwen3-VL-2B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct) fp16 約 4.5GB，T4 可以執行。

In [ ]:
import gc
del siglip; gc.collect(); torch.cuda.empty_cache()

from transformers import AutoModelForImageTextToText
vlm_id = "Qwen/Qwen3-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(vlm_id)
vlm = AutoModelForImageTextToText.from_pretrained(vlm_id, dtype=torch.float16, device_map="auto")

In [ ]:
def ask(images, question, max_new_tokens=256):
    images = images if isinstance(images, list) else [images]
    messages = [{"role": "user", "content": [{"type": "image", "image": im} for im in images] +
                                            [{"type": "text", "text": question}]}]
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt").to(vlm.device)
    out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

In [ ]:
img = Image.open(urlopen("https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg")).convert("RGB")
show(img)
print(ask(img, "請用繁體中文詳細描述這張圖片。"))

In [ ]:
print(ask(img, "圖中的狗是什麼品種？牠在做什麼？"))

In [ ]:
show(images[0], 200)
print(ask(images[0], "這是什麼動物？什麼品種？用一句話回答。"))
print("答案:", class_names[labels[0]])

### 文字辨識 (OCR) 與資訊擷取

我們自己畫一張收據來測試。

In [ ]:
receipt = Image.new("RGB", (520, 400), "white")
draw = ImageDraw.Draw(receipt)
font = ImageFont.load_default(size=26)
lines = ["SUNSHINE CAFE", "2026-11-10  14:32", "", "Latte        x2   $9.00", "Bagel        x1   $3.50",
         "Cheesecake   x1   $5.25", "", "Subtotal         $17.75", "Tax 8%            $1.42", "TOTAL            $19.17"]
for i, line in enumerate(lines):
    draw.text((30, 20 + i * 36), line, fill="black", font=font)
show(receipt, 350)

In [ ]:
print(ask(receipt, "這張收據是哪家店的？總共多少錢？買了哪些東西？"))

### 多張圖片

In [ ]:
print(ask([images[1], images[2]], "這兩張圖片中的動物有什麼相同和不同的地方？"))

## Q
讓 VLM 對收據輸出 JSON 格式（店名、日期、品項清單含數量與價格、總金額），並用 `json.loads` 解析。
再用程式驗算品項加總是否等於 Subtotal。

In [ ]:
# %load q_vlm_json.py
%run -i q_vlm_json.py